# Marginal versus joint smoothing

This compact tutorial fits the same dynamic diffusion model with Superstats' two smoothing approximations. The marginal approximator estimates each $v_t$ conditional on the observations and sampled invariants. The joint approximator additionally conditions each $v_t$ on the preceding trajectory.

In [ ]:
import bayesflow as bf
import matplotlib.pyplot as plt
import numpy as np

import superstats as sup

In [ ]:
NUM_STEPS = 50
TRAIN_SIZE = 20_000
VALIDATION_SIZE = 250
TEST_SIZE = 250
EPOCHS = 50
BATCH_SIZE = 32
NUM_SAMPLES = 250

## Model and shared simulations

In [ ]:
joint_prior = sup.JointPrior(
    v=sup.transition.RandomWalk(bounds=(-6.0, 6.0)),
    a=sup.transition.Linear(
        bounds=(0.2, 4.0),
        intercept=sup.Prior("normal", loc=2.0, scale=0.5),
        slope=sup.Prior("normal", loc=0.0, scale=1.0),
    ),
    tau=sup.Prior("halfnormal", scale=0.5),
    bias=0.5,
)

model = sup.Model(
    prior=joint_prior,
    simulator=sup.simulation.sample_ddm,
    missing=None,
    contamination=None,
)

In [ ]:
train_data = model.sample(batch_size=TRAIN_SIZE, num_steps=NUM_STEPS)
validation_data = model.sample(batch_size=VALIDATION_SIZE, num_steps=NUM_STEPS)
test_data = model.sample(batch_size=TEST_SIZE, num_steps=NUM_STEPS)

## Approximators

The string interface selects the approximation family. Both use Superstats' depth-2 spline coupling flows. Joint smoothing uses BayesFlow's default autoregressive transformer encoder and decoder; marginal smoothing uses the Superstats recurrent defaults.

In [ ]:
marginal_workflow = sup.Workflow(
    model=model,
    approximator="marginal",
    mode="smoothing",
)
joint_workflow = sup.Workflow(
    model=model,
    approximator="joint",
    mode="smoothing",
)

workflows = {"Marginal": marginal_workflow, "Joint": joint_workflow}

## Train and sample

Both workflows see the same simulations for the same number of optimizer updates. BayesFlow applies its default warm-started cosine-decay learning-rate schedule.

In [ ]:
histories = {}
for name, workflow in workflows.items():
    print(f"Training {name.lower()} smoother")
    histories[name] = workflow.fit_offline(
        data=train_data,
        validation_data=validation_data,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        save_history=False,
        verbose=2,
    )

In [ ]:
posterior_samples = {
    name: workflow.sample(test_data, num_samples=NUM_SAMPLES, batch_size=4)
    for name, workflow in workflows.items()
}

## Time-varying verification

The same held-out trajectories are used for both diagnostic figures.

In [ ]:
metric_summary = {}
for name, workflow in workflows.items():
    estimates = posterior_samples[name]["v"]
    targets = test_data["v"]
    metric_summary[name] = {
        "correlation": round(float(np.mean(sup.diagnostics.correlation_per_step(estimates, targets))), 3),
        "nRMSE": round(float(np.mean(sup.diagnostics.nrmse_per_step(estimates, targets))), 3),
        "contraction": round(float(np.mean(sup.diagnostics.posterior_contraction_per_step(estimates, targets))), 3),
        "calibration error": round(float(np.mean(sup.diagnostics.calibration_error_per_step(estimates, targets))), 3),
    }
    fig = workflow.verify_time_varying(
        targets=test_data,
        estimates=posterior_samples[name],
        variable_keys=["v"],
    )
    fig.suptitle(f"{name} smoothing", y=1.02)
plt.show()

metric_summary

## One estimated trajectory

Both panels show posterior uncertainty for the first held-out trajectory.

In [ ]:
for name, workflow in workflows.items():
    fig = workflow.plot_time_varying_posterior(
        estimates=posterior_samples[name],
        targets=test_data,
        variable_keys=["v"],
        data_idx=0,
        marginal=False,
    )
    fig.suptitle(f"{name} smoothing: one trajectory", y=1.02)
plt.show()